# Task 2: Domain-specific fine-tuning (PDPA compliance triage)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/<gh_user>/CDAZZDEV-MLE-<YourName>/blob/main/task2_genai/task2_genai.ipynb)

QLoRA fine-tune of **Llama-3.2-3B-Instruct** on synthetic data from **gpt-oss-120b** (teacher), judged by **qwen3.8-27b** (third family).
Thin notebook, thick library: every step calls a module in `src/` and prints its result. Every number in the README is rendered from `reports/*.json`.

**Run order and cost**

| Section | Runtime | Spends |
|---|---|---|
| 0 Setup | any | nothing |
| 2A Use case and data | CPU is enough | Groq quota (teacher), resumable |
| 2B Training, merge, push | **T4 GPU** | Colab GPU time, HF upload |
| 2C Evaluation | **T4 GPU** | GPU time, Groq quota (judge) |
| 2D RAG fallback (bonus) | **T4 GPU** | GPU time |
| 2E Package | any | nothing |

Human gates (the notebook stops and asks): **G1** API keys and provider check, **G2** eyeball the first 8 generated items,
**G3** audit 30 examples, **G5** label 20 x 2 outputs, then JP writes two paragraphs of analysis.

Re-running top to bottom is cheap: finished steps load their saved results from `reports/` and Google Drive instead of spending quota again (each cell says so when it does).

## 0. Setup

In [ ]:
# AI-ASSISTED: Claude Code (Claude Opus 5.5), Prompt: 'Thin Colab notebook driving the Task 2 pipeline per plan Phases 0-11', Date: 2026-10-06
# 0.1 Mount Google Drive, find (or clone) the repository, cd into task2_genai
import json
import os
import subprocess
import sys
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/<gh_user>/CDAZZDEV-MLE-<YourName>.git"   # edit once
DRIVE_ROOT = Path("/content/drive/MyDrive/cdazz_task2")   # checkpoints, adapters, merged model, repo copy

try:
    from google.colab import drive, files  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    try:
        drive.mount("/content/drive")
    except Exception as e:   # e.g. VS Code connected to a Colab runtime: Drive auth needs the browser
        print(f"Google Drive not mounted ({type(e).__name__}); using /content/cdazz_task2. "
              "Files there are lost when the runtime ends: download reports/ and data/ (cell 2E.2) before closing.")
        DRIVE_ROOT = Path("/content/cdazz_task2")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    REPO_DIR = DRIVE_ROOT / "repo"
    if not (REPO_DIR / "task2_genai" / "src").exists():
        if "<gh_user>" in GITHUB_REPO_URL:
            raise SystemExit(f"Set GITHUB_REPO_URL above, or upload the repository folder to {REPO_DIR}")
        subprocess.run(["git", "clone", GITHUB_REPO_URL, str(REPO_DIR)], check=True)
    TASK_DIR = REPO_DIR / "task2_genai"
    os.environ["CDAZZ_WORK_DIR"] = str(DRIVE_ROOT)      # checkpoints survive a disconnect
else:
    TASK_DIR = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd() / "task2_genai"

os.chdir(TASK_DIR)
sys.path.insert(0, str(TASK_DIR))


def sh(cmd: str) -> int:
    """Run a shell command from task2_genai/ and print its output."""
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-30000:])
    if r.returncode:
        print(r.stderr[-8000:])
    return r.returncode


print("Working directory:", Path.cwd())

In [ ]:
# 0.2 Install the latest libraries, print versions, freeze the lock file after the first good run
REFRESH_LOCK = False   # True overwrites requirements-colab-lock.txt with this session's versions
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                    "transformers", "trl", "peft", "bitsandbytes", "accelerate", "datasets", "rouge_score", "bert_score",
                    "chromadb", "sentence-transformers", "openai", "pydantic", "pyyaml", "tenacity", "wandb",
                    "huggingface_hub", "tiktoken", "python-dotenv"], check=True)
from src.utils import library_versions

print(json.dumps(library_versions(), indent=1))
lock = Path("requirements-colab-lock.txt")
if IN_COLAB and (REFRESH_LOCK or not lock.exists()):
    lock.write_text(subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout)
    print(f"Wrote {lock} (commit it after the first good run)")

In [ ]:
# 0.3 Secrets: Colab Secrets panel (key icon) -> GROQ_API_KEY, HF_TOKEN, optional WANDB_API_KEY / JUDGE_API_KEY.
# If Colab Secrets are unavailable (for example VS Code connected to a Colab runtime), you are prompted with a hidden
# input box instead. Never type a key into a code cell. Values go into environment variables only; this cell prints
# names and set/missing, never values.
SECRET_NAMES = ["GROQ_API_KEY", "HF_TOKEN", "WANDB_API_KEY", "JUDGE_API_KEY"]
if IN_COLAB:
    from google.colab import userdata
    for _name in SECRET_NAMES:
        try:
            _val = userdata.get(_name)
        except Exception:
            _val = None
        if _val:
            os.environ[_name] = _val
    _val = None
    from getpass import getpass
    for _name in ("GROQ_API_KEY", "HF_TOKEN"):            # required ones; optional keys stay missing if not in Secrets
        if not os.environ.get(_name):
            _typed = getpass(f"{_name} (hidden input, press Enter to skip): ").strip()
            if _typed:
                os.environ[_name] = _typed
            _typed = None
else:
    from dotenv import load_dotenv
    load_dotenv(TASK_DIR.parent / ".env")
print({n: ("set" if os.environ.get(n) else "missing") for n in SECRET_NAMES})
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

In [ ]:
# 0.4 Config, seeds, flags
import pandas as pd
from IPython.display import Image, Markdown, display

from src.config import load_config, load_text
from src.utils import free_memory, read_json, read_jsonl, set_seed, write_json, write_jsonl

cfg = load_config()
set_seed(cfg.project.seed)
pd.set_option("display.max_colwidth", 300)
RERUN_EXPERIMENTS = False   # True recomputes cached GPU experiments (label mask, memory probe, overfit, merge, push)
USE_WANDB = False           # True logs training to Weights & Biases (needs WANDB_API_KEY)
if USE_WANDB and os.environ.get("WANDB_API_KEY"):
    import wandb
    wandb.login(key=os.environ["WANDB_API_KEY"])
    os.environ["WANDB_PROJECT"] = "cdazz-task2"
    cfg.training.report_to = "wandb"
print("Checkpoint/work dir:", cfg.work_dir())
print("Teacher:", cfg.teacher.model, "| Student:", cfg.student.model_id, "| Judge:", cfg.judge.model)

In [ ]:
# 0.5 Unit tests (CPU): schemas, JSON extraction, seeds, filters, split, metrics, training utilities
sh("python -m pytest -q")

## 2A. Use case and data

### Problem statement

Task: PDPA compliance triage.

Input: one internal request or incident note from an employee of a fictional Sri Lankan bank ("the Bank"), 50 to 240 words, describing a proposed or ongoing activity that involves personal data.

Output: one JSON object with keys in this order:

```json
{
  "rationale": "2 to 4 sentences reasoning from the facts to the verdict",
  "principles": ["P15", "P02"],
  "issues": ["concrete problem tied to a stated fact"],
  "missing_information": [],
  "verdict": "COMPLIANT | NON_COMPLIANT | NEEDS_MORE_INFO",
  "risk_level": "LOW | MEDIUM | HIGH",
  "required_actions": ["verb-first concrete action"]
}
```

Rationale comes first on purpose: for a 3B model, writing the reasoning before the decision gives it tokens to think with.

A response is correct when:

- it is valid JSON with exactly these keys and allowed values,
- the verdict matches what a careful reviewer concludes from the scenario and the rulebook,
- the risk level follows the rulebook's risk guide,
- the principles include the primary relevant principle and nothing clearly irrelevant,
- every issue and action is grounded in facts stated in the scenario,
- it states no invented facts, principle IDs, Act sections, fines or deadlines.

A response is incorrect when the verdict is wrong, or it is unparseable. It is hallucinated when it asserts anything not supported by the scenario or rulebook (full labelling guide in Appendix F).

### Models: three different families on purpose

| Role | Model | Where | Why |
|---|---|---|---|
| Teacher (data generator) | `openai/gpt-oss-120b` | Groq free tier | Strongest free model on Groq today. Supports strict JSON schema (constrained decoding) and prompt caching. |
| Student (fine-tuned) | `meta-llama/Llama-3.2-3B-Instruct` (ungated mirror with identical weights: `unsloth/Llama-3.2-3B-Instruct`) | Colab T4 | Different family from teacher. 3B fits a T4 with room for batch 4 at about 1.2K tokens, merges in fp16 on the GPU, trains fast enough to run a small sweep inside free quota. |
| Judge (LLM-as-judge) | `qwen/qwen3.8-27b` | Groq free tier (preview) | Third family. A judge from the teacher's family would favour the fine-tuned model, because the fine-tuned model imitates the teacher's style. |

What changed from the brief's suggestions (say this in the README, it shows you checked):

- Groq's live model page on 2026-10-06 lists Llama 3.3 70B as Enterprise only. The free plan chat models are gpt-oss-120b, gpt-oss-20b and qwen3.8-27b. So "Groq Llama-3-70B" as teacher is no longer a free option.
- Free plan limits on 2026-10-06: gpt-oss-120b and qwen3.8-27b are each 30 RPM, 1K requests/day, 8K tokens/min, 200K tokens/day. Cached prompt tokens do not count toward limits, and caching works on gpt-oss models only.

Why 3B and not 7B: a 7B model in fp16 is about 14 GB. The T4 has about 15 GB and Colab free has about 12.7 GB of system RAM, so merging a 7B in fp16 is fragile on free Colab. A 3B model gives the same scientific story with time left for a learning-rate sweep, an overfit test and proper evaluation. Smaller model, better data, honest eval.

Fallbacks (config-driven, no code changes):

- Student: `microsoft/Phi-4-mini-instruct` if Llama access or fp16 behaviour is a problem.
- Teacher top-up: `openai/gpt-oss-20b` (separate 200K/day quota), capped at 25 percent of the dataset, tagged per example.
- Judge: if qwen3.8-27b (a preview model) is withdrawn, use a free model from another family through any OpenAI-compatible endpoint (for example Mistral's free tier). Last resort gpt-oss-20b, with the teacher-family bias stated in the README.

Licence note: the Llama 3.2 Community License requires "Llama" at the start of the name of a distributed derivative model and a "Built with Llama" notice. So the Hub repo is `Llama-3.2-3B-PDPA-Triage` and the model card says "Built with Llama".

### 2A.1 Rulebook and prompts

The rulebook (`knowledge/pdpa_policy_rulebook.md`) is a fictional bank policy modelled on Sri Lanka's PDPA (Act No. 9 of 2022, amended 2025).
It has no section numbers, so any cited section number is a hallucination by definition. The teacher prompt below is printed **in full**.

In [ ]:
# 2A.1 Student prompt and the full rendered teacher prompt (token count must exceed the provider's minimum cacheable length)
from src.generate_data import count_tokens, render_teacher_system_prompt, save_rendered_teacher_prompt

print("=== STUDENT SYSTEM PROMPT (prompts/student_system_prompt.md) ===\n")
print(load_text(cfg.prompt_path("student_system_prompt.md")))
teacher_prompt = render_teacher_system_prompt(cfg)
save_rendered_teacher_prompt(cfg)
n_tok, method = count_tokens(teacher_prompt)
write_json(cfg.report_path("teacher_prompt_tokens.json"), {"tokens": n_tok, "method": method})
print(f"=== TEACHER SYSTEM PROMPT, rendered with the rulebook: {n_tok} tokens ({method}) ===\n")
print(teacher_prompt)

### 2A.2 Gate G1: provider check

Needs `GROQ_API_KEY`. Lists the served models, sends one tiny strict-schema request to the teacher and the judge with the reasoning
settings from `config.yaml`, and prints usage and `x-ratelimit-*` headers. If a call returns 400, fix `config.yaml` (model id,
`strict_schema`, `extra_body`) following Groq's docs, not the code.

In [ ]:
# 2A.2 Provider check (gate G1)
sh("python scripts/provider_check.py")

In [ ]:
# 2A.3 Stratified seeds (400): categorical dimensions cycled and shuffled independently, weighted dimensions by exact quotas
from src.seeds import build_seeds, coverage

if not cfg.path("seeds").exists():
    write_jsonl(cfg.path("seeds"), build_seeds(cfg))
seeds = read_jsonl(cfg.path("seeds"))
print(f"{len(seeds)} seeds in {cfg.path('seeds')}")
for dim, counts in coverage(seeds).items():
    print(f"{dim}: min {min(counts.values())}, max {max(counts.values())} over {len(counts)} values")
display(pd.DataFrame(seeds).head(8))

### 2A.4 Teacher generation

Budget maths (Groq free plan, checked 2026-10-06; these are plan estimates, the measured numbers are in the log summary below):
per call (4 examples) about 400 uncached input tokens, 1,700 output tokens and 300 reasoning tokens, so about 2.4K counted tokens
because the ~3K-token system prefix is cached and cached tokens do not count. 200K tokens/day gives about 80 calls (about 320 examples)
per day; 8K TPM allows about 3 calls per minute, so a day's quota takes about 30 minutes. If the cache hit rate is near zero, stop and fix
the prefix before burning quota.

**Gate G2:** the next cell makes a 2-call smoke run (8 items) and shows them. Read them before running the full generation.

In [ ]:
# 2A.4a Smoke run: 2 calls (8 items), then show them for gate G2
from src.generate_data import generation_log_summary, items_table, run_generation

if not read_jsonl(cfg.path("generations")) and not read_jsonl(cfg.path("rejected")):
    run_generation(cfg, max_calls=2)
else:
    print("Generation already started earlier: no API calls made in this cell (showing the first 8 accepted items).")
display(pd.DataFrame(items_table(read_jsonl(cfg.path("generations"))[:8])))
print(json.dumps(generation_log_summary(cfg), indent=1))

In [ ]:
# 2A.4b Full generation (resumable; stops cleanly at the daily token budget; re-run on the next day to continue)
TARGET_ACCEPTED = 300          # plan target; floor 200 (still 2x the brief's minimum)
RUN_TEACHER = True             # False never calls the API in this cell
USE_FALLBACK_TEACHER = False   # True tops up with gpt-oss-20b (capped at 25% of seeds, tagged per item)
RETRY_REJECTED = False         # True re-attempts seeds that were rejected before (use after all seeds were tried once)

summary = generation_log_summary(cfg)
if summary["accepted"] >= TARGET_ACCEPTED or not RUN_TEACHER:
    print(f"Loaded cached data: {summary['accepted']} accepted items already on disk. "
          "No API calls made and no quota spent in this cell.")
else:
    teacher = cfg.teacher_fallback.model if USE_FALLBACK_TEACHER else cfg.teacher.model
    print(json.dumps(run_generation(cfg, model=teacher, retry_rejected=RETRY_REJECTED), indent=1))
summary = generation_log_summary(cfg)
write_json(cfg.report_path("generation_summary.json"), summary)
print(json.dumps(summary, indent=1))

In [ ]:
# 2A.5 Clean: schema/consistency -> leakage -> forbidden content -> exact dups -> near dups (MiniLM cosine >= 0.92)
from src.validate_data import run_filters

clean, filter_report = run_filters(cfg, read_jsonl(cfg.path("generations")))
write_jsonl(cfg.path("clean"), clean)
write_json(cfg.report_path("filter_report.json"), filter_report)
display(pd.DataFrame([{k: v for k, v in s.items()} for s in filter_report["stages"]]))
print("Clean items:", filter_report["output"], filter_report["output_by_verdict"])

### 2A.6 Gate G3: look at the data

The next cell writes 30 random clean items to `data/audit_log.md` as cards (it never overwrites an existing audit).
JP reads all 30 and fills the four lines under each card. **If more than 3 of 30 labels are wrong, fix the teacher prompt and
regenerate the affected slice.** Re-run the cell after the audit to count the answers.

In [ ]:
# 2A.6 Audit cards (gate G3) and the audit summary
from src.validate_data import audit_summary, write_audit

if not cfg.path("audit_log").exists():
    print("Wrote", write_audit(cfg, cfg.filters.audit_n), "- open it, fill every card, then re-run this cell.")
audit = audit_summary(cfg)
write_json(cfg.report_path("audit_summary.json"), audit)
print(json.dumps(audit, indent=1))
if not audit["complete"]:
    print("Audit not finished yet: fill data/audit_log.md before training.")

In [ ]:
# 2A.7 Diversity report: lengths, coverage, keywords, principle co-occurrence, distinct-n, nearest-neighbour cosine
from src.diversity import run as diversity_run

div = diversity_run(cfg)
print(json.dumps({k: div[k] for k in ("n_items", "lengths_words", "lexical")}, indent=1))
print(div.get("verdict_line", ""))
display(pd.DataFrame(div["category_keyword_hit_rate"]).T)
for fig in ("length_histograms.png", "coverage.png", "keywords.png", "principle_cooccurrence.png", "nn_cosine_hist.png"):
    if cfg.figure_path(fig).exists():
        display(Image(str(cfg.figure_path(fig))))

In [ ]:
# 2A.8 Stratified 80/10/10 split (seed 42), cross-split leakage check, chat JSONL; one example through the real Llama 3.2 template
from src.split_format import run as split_run
from src.train import load_tokenizer
from src.utils import chat_text

split_report = split_run(cfg)
print(json.dumps({k: split_report[k] for k in ("total", "sizes", "fractions", "verdict_by_split")}, indent=1))
print("Leakage check moved to train:", split_report["leakage_check"]["n_moved"])
example = read_jsonl(cfg.split_path("train"))[0]
print("\n=== One training record (JSONL) ===")
print(json.dumps(example, ensure_ascii=False, indent=1)[:4000])
tok = load_tokenizer(cfg)
print("\n=== Rendered with tokenizer.apply_chat_template (pinned date, identical at train and inference) ===")
print(chat_text(tok, example["messages"], add_generation_prompt=False))

In [ ]:
# 2A.9 Render the data card and README tables from reports/
sh("python scripts/render_results.py")

## 2B. QLoRA training on a Colab T4

Switch the runtime to **T4 GPU** and re-run Section 0 first. Checkpoints go to Google Drive, so a disconnect resumes from the
last epoch (`resume_from_checkpoint`), and finished runs are reused instead of retrained.

Optional plumbing check before spending GPU time: set `RUN_SMOKE = True` below (SmolLM2-135M, 16 examples, 1 epoch).

In [ ]:
# 2B.0 Optional smoke test (same train -> infer -> metrics code path with a tiny model)
RUN_SMOKE = False
if RUN_SMOKE:
    sh("python scripts/smoke_cpu_train.py")

In [ ]:
# 2B.1 GPU check (T4: native bf16 unsupported, so fp16 compute)
import torch

from src.train import gpu_check

info = gpu_check()
print(info.pop("nvidia_smi"))
print(info)
assert info["cuda_available"], "Switch the runtime to a GPU (Runtime > Change runtime type > T4 GPU)."
CKPT = cfg.work_dir() / "checkpoints"
CKPT.mkdir(parents=True, exist_ok=True)
print("Checkpoints:", CKPT)

In [ ]:
# 2B.2 Tokenizer: pad = <|finetune_right_pad_id|> (never eos), right padding for training, pinned template date
tok = load_tokenizer(cfg)
print("pad:", tok.pad_token, tok.pad_token_id, "| eos:", tok.eos_token, tok.eos_token_id, "| padding side:", tok.padding_side)
assert tok.pad_token_id != tok.eos_token_id, "pad must not be eos, or the model never learns to stop"
print("Template date pinned:", "strftime_now" not in tok.chat_template)

In [ ]:
# 2B.3 Token stats with the real chat template -> data-driven max_length (zero truncated examples)
from src.train import token_stats

stats = token_stats(cfg, tok)
write_json(cfg.report_path("token_stats.json"), stats)
MAX_LENGTH = stats["max_length"]
print(json.dumps(stats, indent=1))

In [ ]:
# 2B.4 Label-mask check: decode one real training batch; supervised positions must be exactly the answer JSON + <|eot_id|>
from src.train import inspect_label_mask

p = cfg.report_path("label_mask_check.json")
if p.exists() and not RERUN_EXPERIMENTS:
    mask = read_json(p)
    print(f"(loaded from {p}; set RERUN_EXPERIMENTS=True to recompute)")
else:
    mask = inspect_label_mask(cfg, tok, MAX_LENGTH, CKPT)
    write_json(p, mask)
print("=== FULL INPUT (decoded input_ids) ===\n", mask["full_text"])
print("\n=== SUPERVISED ONLY (labels != -100) ===\n", mask["supervised_text"])
print({k: mask[k] for k in ("n_tokens", "n_supervised_tokens", "ends_with_eot", "supervised_is_valid_answer_json", "passed")})
assert mask["passed"], "completion-only loss is not set up correctly"

In [ ]:
# 2B.5 Memory probe and OOM log: batch 16 without checkpointing vs the chosen config, on the longest examples
from src.train import memory_probe

if cfg.report_path("memory_probe.json").exists() and not RERUN_EXPERIMENTS:
    print(f"(loaded from reports/memory_probe.json; set RERUN_EXPERIMENTS=True to recompute)")
else:
    memory_probe(cfg, tok, MAX_LENGTH, CKPT)
print(cfg.report_path("oom_log.md").read_text())

In [ ]:
# 2B.6 Overfit test: 8 examples, 60 steps, LR 2e-4. Loss must fall below 0.05 or the pipeline is broken.
from src.train import overfit_test

if cfg.report_path("overfit_test.json").exists() and not RERUN_EXPERIMENTS:
    ov = read_json(cfg.report_path("overfit_test.json"))
    print("(loaded from reports/overfit_test.json)")
else:
    ov = overfit_test(cfg, tok, MAX_LENGTH, CKPT)
print({k: v for k, v in ov.items() if k != "losses"})
display(Image(str(cfg.figure_path("overfit_test.png"))))
assert ov["passed"], "Overfit test failed: debug before the sweep."

### 2B.8 Hyperparameters (every value with a reason)

`configs/config.yaml` carries the same values with a one-line comment each. Data-driven values (max length, warmup steps, the chosen learning rate) are printed by the cells around this one and rendered in the README from `reports/`.

| Parameter | Value | Reason |
|---|---|---|
| Base model | Llama-3.2-3B-Instruct | Different family from teacher and judge; fits T4 for QLoRA and for fp16 merge; instruct tuned, so the zero-shot baseline is meaningful. |
| Quantisation | 4-bit NF4 | Required by the brief. NF4 is designed for normally distributed weights (QLoRA paper) and cuts base weights to about a quarter of fp16. |
| Double quantisation | True | Also quantises the quantisation constants; saves roughly 0.4 bits per parameter for free. |
| Compute dtype | float16 | T4 (Turing) has no native bf16, so fp16 is the fast correct choice. |
| LoRA rank r | 16 | Task is format plus policy mapping over about 300 examples. r=8 risks underfitting the 16-way principle selection; r=64 adds capacity we cannot fill and overfits faster. 16 is the middle ground; trainable params printed. |
| LoRA alpha | 32 | Scaling alpha/r = 2, a common stable setting; keeps update size sensible at r=16. |
| LoRA dropout | 0.1 | QLoRA paper used 0.1 for models up to 13B; small dataset needs regularisation. |
| Target modules | q, k, v, o, gate, up, down projections | QLoRA paper found adapting all linear layers is needed to match full fine-tuning quality; attention-only LoRA underperforms. |
| LoRA bias | none | Standard; training biases adds little and complicates merging. |
| Learning rate | 2e-4 (picked from {1e-4, 2e-4} by val loss) | QLoRA paper default for small models; confirmed by sweep, not assumed. Final value comes from `sweep.csv`. |
| LR scheduler | cosine | Smoothly lowers the LR in the last epoch, which helps validation loss keep falling instead of bouncing. |
| Warmup | 5% of total steps (computed, about 3 steps) | Avoids a large first update on fresh LoRA weights; short because total steps are few. |
| Epochs | 3 | About 16 optimiser steps per epoch at this data size; 3 epochs (about 48 steps) is enough to learn the format and policy mapping. More epochs start memorising (val loss turns up). |
| Per-device train batch size | 4 | Largest size that fits the T4 with checkpointing and headroom, from the memory probe. |
| Gradient accumulation | 4 | Effective batch 16 for stabler gradients on a small, varied dataset. |
| Max sequence length | data-driven (expected 1152 or 1280) | Set to the longest example rounded up to 128, so nothing is truncated (truncation would cut the end-of-turn token). |
| Loss | completion only | Training on the system prompt and scenario wastes capacity and inflates loss with tokens we never generate. |
| Packing | False | Examples are short and few; packing complicates per-example completion masking and makes per-epoch loss harder to read. |
| Optimiser | paged_adamw_8bit | Paged optimiser from the QLoRA paper; 8-bit states save memory and avoid OOM spikes. |
| Weight decay | 0.0 | QLoRA paper setting; regularisation comes from dropout and few epochs. |
| Max grad norm | 0.3 | QLoRA paper setting; guards against spikes in fp16. |
| Gradient checkpointing | True (use_reentrant False) | Trades compute for memory so batch 4 fits; non-reentrant mode is the recommended path with PEFT. |
| fp16 / bf16 | True / False | Matches T4 hardware. |
| Eval and save strategy | epoch (plus step-0 eval) | Rubric asks for per-epoch losses; step 0 gives the starting point. |
| load_best_model_at_end | True, metric eval_loss | Safety net if the last epoch is not the best; with monotonic decrease it is the last epoch anyway. |
| save_total_limit | 3 | Keeps Drive usage small. |
| Logging steps | 2 | About 8 points per epoch, enough to average a per-epoch train loss. |
| Seed / data_seed | 42 / 42 | Reproducibility. |
| Eval batch size | 4 | Same memory budget as training. |
| group_by_length | False | Dataset is small; random order is simpler to reason about. |
| torch_compile | False | Not reliable with 4-bit layers on T4; not worth the risk. |
| NEFTune | not used | Keeps the effect of fine-tuning attributable to data and LoRA alone. |
| Generation (eval) | greedy, max_new_tokens 512 | Deterministic comparisons across systems; 512 covers the longest reference with margin. |
| Teacher sampling | temperature 1.0, top_p 1.0, reasoning low | Diversity from sampling plus seeds; quality enforced by strict schema and validators; low reasoning saves daily tokens. |
| Judge sampling | temperature 0, reasoning none | Repeatable scores; instruct mode is cheaper and enough for rubric scoring. |

In [ ]:
# 2B.7 LR sweep on validation loss only: lr in {1e-4, 2e-4}, 3 epochs, step-0 eval. Fallbacks: 2 epochs, then dropout 0.15.
# Each run takes roughly 30-40 min on a T4 (measured times are in sweep.csv). Finished runs are reused, not retrained.
from src.train import lr_sweep

sweep = lr_sweep(cfg, tok, MAX_LENGTH, CKPT / "sweep")
display(pd.read_csv(cfg.report_path("sweep.csv")))
for n in sweep["notes"]:
    print("NOTE:", n)

In [ ]:
# 2B.8 Final run = the chosen sweep run (not retrained for show): per-epoch table and loss curves
from src.train import is_strictly_decreasing

ts = read_json(cfg.report_path("training_summary.json"))
print("Chosen:", ts["chosen_run"], "|", ts["selection_rule"])
print({k: ts["chosen"][k] for k in ("learning_rate", "epochs", "lora_dropout", "warmup_steps", "max_length",
                                     "steps_per_epoch", "trainable_params", "wall_time_s", "peak_gpu_gb", "gpu")})
display(pd.DataFrame(ts["table"]))
print("Validation loss strictly decreasing at every epoch (0 -> last):", is_strictly_decreasing([r["val_loss"] for r in ts["table"]]))
display(Image(str(cfg.figure_path("loss_curves.png"))))
ADAPTER_DIR = Path(ts["chosen"]["adapter_dir"])
print("Adapter:", ADAPTER_DIR)

In [ ]:
# 2B.9 Push the LoRA adapter to the Hub
from src.merge_push import push_adapter, push_merged, repo_ids

links_path = cfg.report_path("hub_links.json")
links = read_json(links_path) if links_path.exists() else {}
if "LoRA adapter" not in links or RERUN_EXPERIMENTS:
    links["LoRA adapter"] = push_adapter(cfg, ADAPTER_DIR)
    write_json(links_path, links)
print(links)

In [ ]:
# 2B.10 Merge: fp16 base on GPU + adapter -> merge_and_unload -> save -> parity check -> push (public)
from src.merge_push import merge_adapter, parity_check

free_memory()
MERGED_DIR = cfg.work_dir() / "merged" / cfg.hub.merged_repo
if not (MERGED_DIR / "config.json").exists() or RERUN_EXPERIMENTS:
    merge_adapter(cfg, ADAPTER_DIR, MERGED_DIR)
if cfg.report_path("merge_parity.json").exists() and not RERUN_EXPERIMENTS:
    parity = read_json(cfg.report_path("merge_parity.json"))
    print("(parity loaded from reports/merge_parity.json)")
else:
    parity = parity_check(cfg, ADAPTER_DIR, MERGED_DIR)
print(json.dumps({k: v for k, v in parity.items() if not k.startswith("outputs")}, indent=1))
if "Merged model" not in links or RERUN_EXPERIMENTS:
    links["Merged model"] = push_merged(cfg, MERGED_DIR)
    write_json(links_path, links)
print(links)

In [ ]:
# 2B.11 Evaluate what you ship: reload the merged model fresh from the Hub and run one test example
from src.merge_push import fresh_reload_check

row = fresh_reload_check(cfg)[0]
print("Model:", repo_ids(cfg)["merged"])
print(row["raw_text"])
print({k: row[k] for k in ("parse_ok", "schema_ok", "generated_tokens", "perplexity", "finish_reason")})

## 2C. Evaluation

Same test split, same student system prompt, greedy decoding, `max_new_tokens=512`, fp16 weights, batched with left padding.
Systems: majority baseline, base zero-shot (the rubric baseline), base 3-shot (fairer baseline: one fixed train example per verdict),
fine-tuned (merged model from the Hub). Predictions are saved, so re-running reuses them.

In [ ]:
# 2C.1 Base model: zero-shot and 3-shot
from src.infer import load_generation_tokenizer, load_model_fp16, run_system

free_memory()
tok_g = load_generation_tokenizer(cfg, cfg.student.model_id)
base = load_model_fp16(cfg.student.model_id, cfg, tok_g)
for system in ("base_zeroshot", "base_fewshot"):
    run_system(cfg, system, base, tok_g)
del base
free_memory()

In [ ]:
# 2C.2 Fine-tuned model (merged, loaded from the Hub)
from src.infer import load_generation_tokenizer, load_model_fp16, run_system
from src.merge_push import repo_ids

FT_MODEL = repo_ids(cfg)["merged"]
tok_ft = load_generation_tokenizer(cfg, FT_MODEL)
ft = load_model_fp16(FT_MODEL, cfg, tok_ft)
run_system(cfg, "ft", ft, tok_ft)
del ft
free_memory()

In [ ]:
# 2C.3 Metrics: ROUGE-L (full / rationale / canonical), BERTScore, task metrics, paired bootstrap CIs (10,000 resamples)
from src.metrics import comparison_table
from src.metrics import run as metrics_run

res = metrics_run(cfg)
display(Markdown(comparison_table(res)))

### 2C.4 LLM-as-judge

Blind and pointwise (one candidate per call, never the system name, shuffled with a fixed seed), temperature 0, strict `JudgeScore`
schema validated with Pydantic (invalid replies retried twice, then excluded and counted). Zero-shot and fine-tuned are judged first
(about 70 calls), few-shot only if the day's quota remains. Resumable: re-run the next day to finish.

In [ ]:
# 2C.4 Judge, then recompute the table with judge scores
from src.judge import run_judge

run_judge(cfg)
res = metrics_run(cfg)
display(Markdown(comparison_table(res)))
if "judge" in res:
    display(pd.DataFrame(res["judge"]["systems"]).T)
    print(json.dumps(res["judge"]["comparisons"], indent=1))

In [ ]:
# 2C.5 Figures: key metrics by system, verdict confusion matrices; accuracy by verdict and difficulty
for fig in ("metrics_bar.png", "confusion_matrices.png"):
    display(Image(str(cfg.figure_path(fig))))
display(pd.DataFrame({s: {g: v["accuracy"] for g, v in d.items()} for s, d in res["per_verdict_accuracy"].items()}))
display(pd.DataFrame({s: {g: v["accuracy"] for g, v in d.items()} for s, d in res["per_difficulty_accuracy"].items()}))

### 2C.6 Gate G5: manual review (JP labels; Claude Code never fills labels)

The next cell writes `reports/manual_review.csv`: 20 test items (stratified by verdict, seed 42) x 2 systems (fine-tuned and base
zero-shot), shuffled, with the system hidden in `reports/manual_review_key.csv`. Do not open the key until labelling is done.

Labelling guide (plan Appendix F), precedence top to bottom:
1. `hallucinated`: states anything not supported by the scenario or rulebook (invented fact, principle ID, rule, Act section, fine, penalty, deadline or authority).
2. `partially_correct`: nothing invented, but wrong verdict, risk off by a level, a principle missed or irrelevant one added, vague actions, or broken format with correct content.
3. `correct`: verdict and risk follow the guides, principles include the primary one, every issue and action grounded, valid JSON.

`error_type` (semicolon separated): none; wrong_verdict; wrong_risk; missed_principle; irrelevant_principle; invented_fact;
invented_id_or_law; vague_actions; format_error; reference_wrong.

In [ ]:
# 2C.6 Build the blind labelling sheet (refuses to overwrite a sheet that already has labels)
from src.manual_review import build_sheet

if not cfg.report_path("manual_review.csv").exists():
    build_sheet(cfg)
else:
    print("reports/manual_review.csv already exists; not rebuilt.")

In [ ]:
# 2C.7 Score JP's labels: hallucination rate with Wilson 95% CI, wrong verdicts, reference_wrong, kappa vs the judge
from src.manual_review import score_sheet, summary_table

review = score_sheet(cfg)
display(Markdown(summary_table(review)))
print(json.dumps({k: review[k] for k in ("rows_labelled", "rows_in_sheet", "complete", "judge_agreement_all",
                                          "teacher_label_noise_estimate")}, indent=1))
for p in review["problems"][:20]:
    print("PROBLEM:", p)

In [ ]:
# 2C.8 Evidence pack for the qualitative analysis (base wrong -> FT right, FT still wrong by error type, accuracy by group)
from src.manual_review import evidence_pack

display(Markdown(evidence_pack(cfg)))

### 2C.9 Qualitative analysis

_To be written by JP from the evidence pack above: two paragraphs citing example IDs._

_Paragraph 1: where fine-tuning helped._

_Paragraph 2: remaining failure modes and the fix for each._

## 2D. Bonus: RAG fallback

Knowledge base: the rulebook only (one chunk per principle subsection plus the verdict and risk guides; never test data) in ChromaDB
with `all-MiniLM-L6-v2` embeddings. Confidence signal: perplexity of the fine-tuned model's own greedy answer. The threshold tau is chosen
on the **validation** set only (maximise the fallback policy's verdict accuracy, floor at the 70th percentile). On test, items with
perplexity > tau or unparseable output are re-asked with the top-3 policy extracts (Appendix E template) and the second answer is used.
Known limitation: the model never saw retrieved context in training, so this is a distribution shift; if RAG does not help, the numbers say so.

In [ ]:
# 2D.1 Build the store, tune tau on validation, route test items, compare before and after
from src.infer import load_generation_tokenizer, load_model_fp16
from src.merge_push import repo_ids
from src.rag_fallback import run_rag

FT_MODEL = repo_ids(cfg)["merged"]

free_memory()
tok_ft = load_generation_tokenizer(cfg, FT_MODEL)
ft = load_model_fp16(FT_MODEL, cfg, tok_ft)
rag = run_rag(cfg, ft, tok_ft)
del ft
free_memory()
print(json.dumps({k: rag[k] for k in ("tau", "gate", "test_size", "routed", "routed_share")}, indent=1))
display(pd.DataFrame({k: rag[k] for k in ("routed_subset_before", "routed_subset_after", "overall_before", "overall_after")}).T)

In [ ]:
# 2D.2 One concrete before-and-after example
ex = rag.get("example")
if ex:
    print(f"Item {ex['id']} (reference verdict {ex['reference_verdict']})\n\nSCENARIO:\n{ex['scenario']}\n")
    print(f"FIRST ANSWER (perplexity {ex['first_perplexity']}):\n{ex['first_answer']}\n")
    print("RETRIEVED CHUNKS:")
    for c in ex["retrieved_chunks"]:
        print(f"  ({c['principle_id']}, {c['subsection']}, distance {c['distance']}) {c['text'][:300]}")
    print(f"\nSECOND ANSWER (perplexity {ex['second_perplexity']}):\n{ex['second_answer']}")
else:
    print("No test item was routed to the fallback at this tau.")

## 2E. Package

Renders every README table from `reports/`, refreshes the Hub model card with the final numbers, scans for secrets, and zips
`data/` + `reports/` for download. Then: File > Download .ipynb, run `python task2_genai/scripts/strip_widget_metadata.py`, commit.

In [ ]:
# 2E.1 Render READMEs, update the model card on the Hub, scan for secrets
from src.merge_push import push_model_card

sh("python scripts/render_results.py")
if os.environ.get("HF_TOKEN"):
    print(push_model_card(cfg, kind="merged"))
sh("python scripts/check_secrets.py")

In [ ]:
# 2E.2 Zip data/ and reports/ (and the rendered prompt) for download into the local repo
import zipfile

archive = Path("/content/task2_outputs.zip") if IN_COLAB else TASK_DIR / "outputs" / "task2_outputs.zip"
archive.parent.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for folder in ("data", "reports"):
        for f in Path(folder).rglob("*"):
            if f.is_file():
                z.write(f, f"task2_genai/{f.as_posix()}")
    for f in ("README.md", "requirements-colab-lock.txt", "prompts/teacher_system_prompt.rendered.md"):
        if Path(f).exists():
            z.write(f, f"task2_genai/{f}")
print("Wrote", archive, f"({archive.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB:
    files.download(str(archive))